In [2]:
import os
if "LOCAL_RANK" not in os.environ:
    os.environ["LOCAL_RANK"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info

/Users/chrisgoerner/miniconda/envs/docling/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from dots_ocr.utils import dict_promptmode_to_prompt

In [12]:
from unittest.mock import patch
from transformers.dynamic_module_utils import get_imports

In [3]:
def inference(image_path, prompt, model, processor):
    # image_path = "demo/demo_image1.jpg"
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path
                },
                {"type": "text", "text": prompt}
            ]
        }
    ]


    # Preparation for inference
    text = processor.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    #inputs = inputs.to("cuda")

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=24000)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    print(output_text)

In [ ]:
# set device
device = "mps" if torch.backends.mps.is_available() else "cpu"

def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    """Work around for https://huggingface.co/microsoft/phi-1_5/discussions/72."""
    print(filename)
    imports = get_imports(filename)
    if not torch.cuda.is_available() and "flash_attn" in imports:
        print("removed")
        imports.remove("flash_attn")
    return imports

In [20]:
import sys
import types

# Create fake flash_attn module so import doesn't crash
flash_attn = types.ModuleType("flash_attn")

def fake_flash_attn_varlen_func(*args, **kwargs):
    raise RuntimeError("flash_attn is not supported on Mac M1 / CPU.")

flash_attn.flash_attn_varlen_func = fake_flash_attn_varlen_func
sys.modules["flash_attn"] = flash_attn

In [24]:
model_path = "/Users/chrisgoerner/Documents/PatientSearch/dots_ocr/weights/DotsOCR_1_5"
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        #attn_implementation="sdpa",
        torch_dtype=torch.bfloat16,
        #torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True
    )
    processor = AutoProcessor.from_pretrained(model_path,  trust_remote_code=True)

/Users/chrisgoerner/Documents/PatientSearch/dots_ocr/weights/DotsOCR_1_5/configuration_dots.py
/Users/chrisgoerner/Documents/PatientSearch/dots_ocr/weights/DotsOCR_1_5/modeling_dots_ocr.py


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.95it/s]


/Users/chrisgoerner/Documents/PatientSearch/dots_ocr/weights/DotsOCR_1_5/configuration_dots.py


In [25]:
image_path = "/Users/chrisgoerner/Documents/PatientSearch/dots_ocr/assets/showcase_dots_ocr_1_5/origin/scene_1.jpg"
for prompt_mode, prompt in dict_promptmode_to_prompt.items():
    print(f"prompt: {prompt}")
    inference(image_path, prompt, model, processor)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


prompt: Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.



KeyboardInterrupt: 